In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.5 Output head and weight tying

> 🎯 **Goal:** Turn the final hidden state into next-token predictions, and learn
> the weight-tying trick that completes PocketLM v1.

**Why this matters.** This is the last piece. After the stack of blocks, the model
still holds a hidden vector per token; the **output head** converts that into a
score for every possible next token. Once this is in place you have the entire
architecture, the model you will train, modernize, adapt, and serve for the rest
of the course.

**The intuition.** The model uses one dictionary for two opposite jobs. When
*reading*, the embedding table looks up 'what does this token mean as a vector?'.
When *writing*, the output head asks the reverse, 'which token does this vector
most look like?'. **Weight tying** says: it is the same dictionary, so use the
same book for both. Reuse one `[vocab, n_embd]` matrix for reading in and writing
out, rather than learning two separate ones.

**The mechanics.** The head is a final `LayerNorm` followed by a linear projection
from `n_embd` to `vocab_size`, producing one **logit** (an unnormalized score) per
vocabulary entry. A linear layer's weight is shaped `[vocab_size, n_embd]`, exactly
the shape of the token embedding table, so PocketLM sets `head.weight` to be the
*same tensor object* as `tok_emb.weight`. That is what `is` checks in the code: not
equal values, but literally one shared array. The payoff is fewer parameters (the
vocab-by-embedding matrix is counted once, not twice) and usually slightly better
results, since the reading and writing views of each token stay consistent. With
this, PocketLM v1 is complete: tokenize, embed, add positions, stack N blocks,
final norm, project to logits.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
print("head tied to embedding:", model.head.weight is model.tok_emb.weight)
print("parameters:", sum(p.numel() for p in model.parameters()))
print(generate(model, tok, "The ", max_new_tokens=40, seed=0))

**What just happened.** We trained a tiny PocketLM end to end and then used it. The
first print confirms `model.head.weight is model.tok_emb.weight` returns `True`,
the head and the embedding really are the one shared matrix, weight tying in
action. The parameter count reflects that saving. The final print is the model
*generating*: given the prompt 'The ' it produces 40 new characters one at a time,
each step feeding its own output back in. It will not be Shakespeare from a model
this small, but it is your architecture running for real.

The asserts lock in both facts: the tie holds, and `generate` returns exactly the
4-character prompt plus 40 new tokens (44 characters total).

That is the full architecture: tokenize, embed, add positions, stack blocks, norm,
project to logits. Everything after this unit is about *training* it well (u5),
*modernizing* it (u6), *adapting* it (u7), and *serving* it (u8).

⚠️ **Common pitfalls**
- Checking the tie with `==` instead of `is`. Equality can be true by coincidence;
  only `is` proves it is the same object whose gradients are shared.
- Forgetting the final `LayerNorm` before the head. Without it the logits sit on an
  unstable scale and training suffers.
- Expecting fluent text from a tiny char-level model. Coherence comes from scale
  and training (u5+), not from the architecture alone.

🏋️ **Try it yourself.** Re-run `generate` with a different prompt and a different
`seed` and watch the output change. Then raise `max_new_tokens` to 120 to see a
longer sample. Finally, recompute the parameter count in your head: tying saves one
`vocab_size * n_embd` matrix, confirm that against the printed total.

In [ ]:
assert model.head.weight is model.tok_emb.weight
assert len(generate(model, tok, "The ", max_new_tokens=40, seed=0)) == 4 + 40